# Example of evaluating electron repulsion integrals (ERIs) of Gausslet orbitals

In [1]:
import numpy as np
from scipy.stats import ortho_group
# see the README for build and install instructions of this package
import gausslectronitegrate as gli

In [2]:
# Gausslet G10 coefficients
import sys
sys.path.append("../gausslets/")
from gausslets import get_gausslet_g10_coeffs
b_list, _ = get_gausslet_g10_coeffs()

In [3]:
# truncation tolerance
# (values close to zero imply higher accuracy but longer computation time)
tol = 1e-16

In [4]:
# compute the overlap integral for all Gausslets centered at the origin
gli.compute_eri_gausslet_integral(b_list, ([0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]), tol)

1.6252009000829197

In [5]:
# evaluate the overlap integral for four other points, as an example
point_a = [ 1,  0,  0]
point_b = [ 0,  0, -1]
point_c = [ 0,  0,  1]
point_d = [ 0,  0,  0]
gli.compute_eri_gausslet_integral(b_list, (point_a, point_b, point_c, point_d), tol)

-0.0005864045861050708

In [6]:
# define a symmetric Cartesian grid by tuples of the form (istart, num points)
# in each coordinate direction
grid = [
    (-1, 3),  # x
    (-1, 3),  # y
    (-1, 3),  # z
]

In [7]:
# overall number of grid points
num_points = gli.grid_num_points(grid)
num_points

27

In [8]:
# evaluate electron repulsion integrals (ERIs) for Gausslet orbitals centered at the grid points,
# taking translational, octahedral and permutational symmetries into account
eri = gli.compute_sparse_eri_gausslet_integrals(b_list, grid, tol)
type(eri)

gausslectronitegrate.SparseERI

In [9]:
# number of stored ERI values after taking symmetries into account
eri.num_entries

1027

In [10]:
# query the integral value for all Gausslets centered at the origin (should agree with above value)
eri.integral_value(([0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]))

1.6252009000829197

In [11]:
# query the integral value at the four other points from above
eri.integral_value((point_a, point_b, point_c, point_d))

-0.0005864045861050727

In [12]:
# as an alternative, load precomputed ERI data from a file
eri_precomp = gli.load_sparse_eri_gausslet_integrals("eri_3x3x3.hdf5")

In [13]:
# precomputed data uses same tolerance
eri_precomp.tol

1e-16

In [14]:
# precomputed data uses same grid specification
eri_precomp.grid

((-1, 3), (-1, 3), (-1, 3))

In [15]:
# precomputed data uses same Gausslet coefficients
np.linalg.norm(eri_precomp.gausslet_coeffs - b_list)

0.0

In [16]:
# accordingly, integral values agree
eri_precomp.integral_value((point_a, point_b, point_c, point_d)) - eri.integral_value((point_a, point_b, point_c, point_d))

0.0

In [17]:
# project ERI tensor onto a basis
rng = np.random.default_rng(293)
# draw 5 random orthogonal state vectors
basis = ortho_group.rvs(num_points, random_state=rng)[:, :5]
eri_tensor_proj = eri.project(basis)
type(eri_tensor_proj)

numpy.ndarray

In [18]:
eri_tensor_proj.shape

(5, 5, 5, 5)

In [19]:
# reference calculation
eri_tensor_dense = eri.fill_tensor(grid)
eri_tensor_proj_ref = np.einsum(
    basis, (4, 0),
    basis, (5, 1),
    basis, (6, 2),
    basis, (7, 3),
    eri_tensor_dense, (4, 5, 6, 7),
    (0, 1, 2, 3), optimize=True)
# difference should be zero (up to rounding errors)
np.linalg.norm(eri_tensor_proj - eri_tensor_proj_ref)

1.0371740181456874e-13

In [20]:
# apply ERI tensor interpreted as a matrix in physicist's convention to a list of states
rng = np.random.default_rng(401)
# draw 3 random two-particle statevectors
states = rng.normal(size=(num_points**2, 3))
eri_states = eri.apply(states)
eri_states.shape

(729, 3)

In [21]:
# reference calculation
# transposition due to integral convention
vint = eri.fill_tensor(grid).transpose((0, 2, 1, 3)).reshape(2*(num_points**2,))
eri_states_ref = vint @ states
# difference should be zero (up to rounding errors)
np.linalg.norm(eri_states - eri_states_ref)

1.3767558695129101e-14